In [ ]:
%load_ext autoreload
%autoreload 2

<!-- SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

# PII Replacement with Plan Review

Run Safe Synthesizer on patient events: load data, optionally review/edit the discovered PII plan, apply replacement, then optionally inspect original vs transformed rows.

#### What you'll learn

- Load `patient_events.csv` into a `SafeSynthesizer` builder
- Optionally review the PII plan as YAML + UML-style cards
- Edit the plan and save/render (validated against the dataset)
- Apply PII replacement with `process_data()`
- Optionally browse original vs transformed records with Anonymizer-style highlights

### Prerequisites

From the repo root:

```bash
uv sync --extra notebook
# Full run also needs the engine + GPU extras used by Safe Synthesizer 101:
# uv sync --extra notebook --extra engine --extra cu129
```


## 1. Load the dataset

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("/root/datasets/patient_events.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns from {DATA_PATH}")
df.head()

## 2. Create the Safe Synthesizer builder

Group events by `patient_id` and order by `event_id` so discovered plans use `scope: group`.

The person generation backend is set to `"faker"` for convenience. If you want to use Nemotron-Personas to sample more realistic-looking synthetic persons, follow [these instructions](https://docs.nvidia.com/nemo/datadesigner/concepts/person-sampling#approach-2-nemotron-personas-datasets) to download the asset(s) for your locale, and do 
```
.with_replace_pii(person={"backend": "managed"})
```

In [ ]:
# Edit this cell to fit your dataset and person generation backend

from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = (
    SafeSynthesizer()
    .with_data_source(df)
    .with_data(
        holdout=0,
        group_training_examples_by="patient_id",
        order_training_examples_by="event_id",
    )
    .with_replace_pii(person={
        "backend": "faker", # "faker" or "managed"
        # "managed_assets_path": # "~/.data-designer/managed-assets",
        })
)
builder

## 3. Review the PII plan (optional)

`review_pii_plan()` discovers (or loads) the plan for this dataframe and opens the editor.
This step is optional — skip it and `process_data()` / `run()` will still auto-discover.

1. Hover cards/rows to highlight YAML (and click in YAML to highlight the diagram).
2. Edit YAML if you want, then click **Save and render diagram** — missing columns and other plan errors surface here.
3. Each successful render syncs the plan onto the builder for the rest of the run.

In [ ]:
editor = builder.review_pii_plan()
editor


## 4. Execute PII Replacement

If you used the editor, the builder now holds that plan (or your last valid edit).
Otherwise this step discovers automatically.

If you want to run all of the remaining pipeline beyond PII replacement, call `builder.run()` instead of `builder.process_data()`.

To iterate, edit the plan above, click **Save and render diagram**, then re-run this cell: the edit
re-arms this step and replacement runs again, refreshing the artifacts in the same run directory.
Re-running this cell without a plan edit is a no-op.

In [ ]:
builder.process_data()
print(f"Training rows after PII replacement: {len(builder._training_df)}")
# builder._training_df[["patient_id", "first_name", "last_name", "provider_name", "notes"]].head()
builder._training_df.head()

# Apply PII and continue into train/generate/evaluate when ready:
# builder.run()

## 5. Preview replaced data (optional)

After replacement, browse changed training rows in one interlaced table: each source column is
shown as an **original** / **transformed** sub-column pair, one row per record. Replaced values get
colored chips, where each original value and the value that replaced it share a color (including
inside free-text columns), and entity types appear under the column names.

If the replacement looks wrong, go back to Step 3, edit the plan, click **Save and render diagram**,
then re-run Step 4 and this cell to see the new result.

In [ ]:
result_preview = builder.preview_replaced_data(max_records=25, only_changed=True)
result_preview

## 6. Run the rest of the pipeline

Training, generation, and evaluation continue from the PII-replaced data. These steps need the
GPU environment from [Safe Synthesizer 101](safe-synthesizer-101.ipynb) and take about 20 minutes,
so the cell below is commented out.

In [ ]:
# builder.train().generate().evaluate().save_results()
# results = builder.results
#
# print(f"Synthetic rows: {len(results.synthetic_data)}")
# results.synthetic_data.head()